# Snowflake Horizon Catalog - End-to-End Demo

This notebook demonstrates the key capabilities of **Snowflake Horizon Catalog** using a synthetic healthcare dataset:

1. **Object Tagging** - Classify data with PII levels, data domains, and retention requirements
2. **Data Governance Policies** - Masking, Row Access, Aggregation, Join, and Projection policies
3. **Lineage** - Track data flow from raw tables through analytics to dashboards
4. **Data Quality Monitoring** - System and custom DMFs for continuous quality checks
5. **Data Sharing & Agentic AI** - Share governed data and AI agents via marketplace

## Architecture

```
Raw Tables (PATIENTS, BILLING, APPOINTMENTS, PRESCRIPTIONS, DOCTORS, INSURANCE_CLAIMS)
    |
    +-- Object Tags (PII_LEVEL, DATA_CLASSIFICATION, DATA_DOMAIN, RETENTION_DAYS)
    +-- Policies (Masking, Row Access, Aggregation, Join, Projection)
    +-- DMFs (NULL_COUNT, DUPLICATE_COUNT, INVALID_SSN_COUNT, etc.)
    |
    v
Staging Views (STG_PATIENTS, STG_BILLING, STG_APPOINTMENTS)
    |
    v
Dynamic Table (PATIENT_ANALYTICS) --> Cortex Agent (HEALTHCARE_AGENT)
    |
    v
Secure Views (REVENUE_DASHBOARD, DEPARTMENT_METRICS) --> Share/Listing
```

---
**Database:** `HORIZON_DEMO_DB.HEALTHCARE`  
**Required Role:** `SYSADMIN` (for setup), demo roles for policy testing

In [ ]:
-- Set context
USE ROLE SYSADMIN;
USE DATABASE HORIZON_DEMO_DB;
USE SCHEMA HEALTHCARE;
USE WAREHOUSE MY_WH;

## 1. Object Tagging

Object tags enable **automated governance at scale**. Instead of manually configuring policies on every column, you tag data by sensitivity level and let tag-based policies handle the rest.

### Tags in this demo:
| Tag | Purpose | Values |
|-----|---------|--------|
| `PII_LEVEL` | Sensitivity classification | HIGH, MEDIUM, LOW, NONE |
| `DATA_CLASSIFICATION` | HIPAA data category | PHI, PII, FINANCIAL, OPERATIONAL, PUBLIC |
| `DATA_DOMAIN` | Business ownership | CLINICAL, FINANCIAL, ADMINISTRATIVE, RESEARCH |
| `RETENTION_DAYS` | Compliance retention | 365, 2555 (7 years for HIPAA) |

In [ ]:
-- View all PII-tagged columns on the PATIENTS table
SELECT
  TAG_NAME,
  TAG_VALUE,
  COLUMN_NAME
FROM TABLE(HORIZON_DEMO_DB.INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
  'HORIZON_DEMO_DB.HEALTHCARE.PATIENTS', 'TABLE'))
WHERE TAG_NAME = 'PII_LEVEL'
ORDER BY
  CASE TAG_VALUE WHEN 'HIGH' THEN 1 WHEN 'MEDIUM' THEN 2 WHEN 'LOW' THEN 3 ELSE 4 END,
  COLUMN_NAME;

In [ ]:
-- Governance report: all tables by data classification
SELECT DISTINCT
  tr.OBJECT_NAME AS table_name,
  tr.TAG_VALUE AS classification,
  CASE tr.TAG_VALUE
    WHEN 'PHI' THEN 'Protected Health Information - HIPAA regulated'
    WHEN 'PII' THEN 'Personally Identifiable Information'
    WHEN 'FINANCIAL' THEN 'Financial/Billing data'
    ELSE 'Other'
  END AS compliance_note
FROM TABLE(HORIZON_DEMO_DB.INFORMATION_SCHEMA.TAG_REFERENCES(
  'HORIZON_DEMO_DB.HEALTHCARE.PATIENTS', 'TABLE')) tr
WHERE tr.TAG_NAME = 'DATA_CLASSIFICATION' AND tr.DOMAIN = 'TABLE'
UNION ALL
SELECT DISTINCT tr.OBJECT_NAME, tr.TAG_VALUE,
  CASE tr.TAG_VALUE WHEN 'PHI' THEN 'Protected Health Information - HIPAA regulated'
    WHEN 'FINANCIAL' THEN 'Financial/Billing data' ELSE 'Other' END
FROM TABLE(HORIZON_DEMO_DB.INFORMATION_SCHEMA.TAG_REFERENCES(
  'HORIZON_DEMO_DB.HEALTHCARE.BILLING', 'TABLE')) tr
WHERE tr.TAG_NAME = 'DATA_CLASSIFICATION' AND tr.DOMAIN = 'TABLE'
UNION ALL
SELECT DISTINCT tr.OBJECT_NAME, tr.TAG_VALUE,
  CASE tr.TAG_VALUE WHEN 'PHI' THEN 'Protected Health Information - HIPAA regulated'
    WHEN 'FINANCIAL' THEN 'Financial/Billing data' ELSE 'Other' END
FROM TABLE(HORIZON_DEMO_DB.INFORMATION_SCHEMA.TAG_REFERENCES(
  'HORIZON_DEMO_DB.HEALTHCARE.APPOINTMENTS', 'TABLE')) tr
WHERE tr.TAG_NAME = 'DATA_CLASSIFICATION' AND tr.DOMAIN = 'TABLE'
ORDER BY classification;

## 2. Data Governance Policies

Five policy types work together to provide defense-in-depth:

| Policy Type | What it does | Applied to |
|------------|--------------|-----------|
| **Masking** | Hides/redacts column values based on role | SSN, Email, Phone, Address |
| **Row Access** | Filters rows based on department role | PATIENTS table |
| **Aggregation** | Forces minimum group size in queries | BILLING table |
| **Join** | Requires analysts to join (no standalone SELECT) | PATIENTS table |
| **Projection** | Prevents SELECT of sensitive columns | PRESCRIPTIONS.MEDICATION_NAME |

### Key design choice: Tag-Based Masking
The masking policy reads the `PII_LEVEL` tag via `SYSTEM$GET_TAG_ON_CURRENT_COLUMN()` to dynamically decide masking level. **One policy protects all tagged columns** - no need to create per-column policies.

In [ ]:
-- DEMO: Masking in action (SYSADMIN sees unmasked, other roles see masked)
-- As SYSADMIN (table owner) - full visibility
SELECT patient_id, first_name, last_name, ssn, email, phone, department
FROM PATIENTS
WHERE department = 'CARDIOLOGY'
LIMIT 5;

In [ ]:
-- View all policies attached to the PATIENTS table
SELECT
  POLICY_NAME,
  POLICY_KIND,
  REF_COLUMN_NAME AS applied_to_column
FROM TABLE(HORIZON_DEMO_DB.INFORMATION_SCHEMA.POLICY_REFERENCES(
  REF_ENTITY_NAME => 'HORIZON_DEMO_DB.HEALTHCARE.PATIENTS',
  REF_ENTITY_DOMAIN => 'TABLE'
))
ORDER BY POLICY_KIND, REF_COLUMN_NAME;

In [ ]:
-- Masking policy definition (tag-based, single policy covers all PII columns)
SELECT GET_DDL('POLICY', 'HORIZON_DEMO_DB.HEALTHCARE.PII_TAG_MASK');

## 3. Lineage

Snowflake automatically tracks data lineage as data flows through views, dynamic tables, and other objects. Use `SNOWFLAKE.CORE.GET_LINEAGE()` to query the lineage graph.

### Our pipeline:
```
PATIENTS ──► STG_PATIENTS ──┐
                            ├──► PATIENT_ANALYTICS (DT) ──► REVENUE_DASHBOARD
APPOINTMENTS ──► STG_APPOINTMENTS ──┤                    └──► DEPARTMENT_METRICS
                            │
BILLING ──► STG_BILLING ────┘
```

In [ ]:
-- DOWNSTREAM LINEAGE: "What depends on the PATIENTS table?"
-- Use case: Impact analysis before schema changes
SELECT
  SOURCE_OBJECT_NAME || ' (' || SOURCE_OBJECT_DOMAIN || ')' AS source,
  '--->' AS flow,
  TARGET_OBJECT_NAME || ' (' || TARGET_OBJECT_DOMAIN || ')' AS target,
  DISTANCE AS hops
FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
  'HORIZON_DEMO_DB.HEALTHCARE.PATIENTS', 'TABLE', 'DOWNSTREAM', 5
))
ORDER BY DISTANCE;

In [ ]:
-- UPSTREAM LINEAGE: "Where does the REVENUE_DASHBOARD data come from?"
-- Use case: Data provenance and trust verification
SELECT
  SOURCE_OBJECT_NAME || ' (' || SOURCE_OBJECT_DOMAIN || ')' AS source,
  '--->' AS flow,
  TARGET_OBJECT_NAME || ' (' || TARGET_OBJECT_DOMAIN || ')' AS target,
  DISTANCE AS hops
FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
  'HORIZON_DEMO_DB.HEALTHCARE.REVENUE_DASHBOARD', 'TABLE', 'UPSTREAM', 5
))
ORDER BY DISTANCE;

In [ ]:
-- COLUMN-LEVEL LINEAGE: "Where does SSN flow in the pipeline?"
-- Use case: PII leakage detection
SELECT
  SOURCE_OBJECT_NAME AS from_table,
  SOURCE_COLUMN_NAME AS from_column,
  TARGET_OBJECT_NAME AS to_table,
  TARGET_COLUMN_NAME AS to_column,
  DISTANCE
FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
  'HORIZON_DEMO_DB.HEALTHCARE.PATIENTS.SSN', 'COLUMN', 'DOWNSTREAM', 5
))
ORDER BY DISTANCE;

## 4. Data Quality Monitoring

Snowflake **Data Metric Functions (DMFs)** run automatically on a schedule and record quality metrics. Results appear in the **Data Governance tab** in Snowsight.

### DMFs attached in this demo:
| Table | DMF | Column | Purpose |
|-------|-----|--------|---------|
| PATIENTS | `NULL_COUNT` | PATIENT_ID | PK should never be null |
| PATIENTS | `DUPLICATE_COUNT` | EMAIL | Emails should be unique |
| PATIENTS | `ROW_COUNT` | (table) | Track growth |
| PATIENTS | `INVALID_SSN_COUNT` (custom) | SSN | Format: XXX-XX-XXXX |
| PATIENTS | `INVALID_PHONE_COUNT` (custom) | PHONE | Format: (XXX) XXX-XXXX |
| APPOINTMENTS | `NULL_COUNT` | PATIENT_ID | FK integrity |
| APPOINTMENTS | `ROW_COUNT` | (table) | Track growth |
| DOCTORS | `DUPLICATE_COUNT` | LICENSE_NUMBER | License must be unique |

In [ ]:
-- View all DMFs attached to PATIENTS
SELECT
  METRIC_DATABASE_NAME || '.' || METRIC_SCHEMA_NAME || '.' || METRIC_NAME AS dmf,
  REF_ARGUMENTS AS monitored_column,
  SCHEDULE,
  SCHEDULE_STATUS
FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
  REF_ENTITY_NAME => 'HORIZON_DEMO_DB.HEALTHCARE.PATIENTS',
  REF_ENTITY_DOMAIN => 'TABLE'
));

In [ ]:
-- Run custom DMFs manually for immediate results
-- (Scheduled runs happen automatically every hour)

-- Check for invalid SSN formats (Expected: 2 from our bad data insert)
SELECT HORIZON_DEMO_DB.HEALTHCARE.INVALID_SSN_COUNT(
  SELECT SSN FROM HORIZON_DEMO_DB.HEALTHCARE.PATIENTS
) AS invalid_ssn_count;

-- Check for invalid phone formats (Expected: 1)
SELECT HORIZON_DEMO_DB.HEALTHCARE.INVALID_PHONE_COUNT(
  SELECT PHONE FROM HORIZON_DEMO_DB.HEALTHCARE.PATIENTS
) AS invalid_phone_count;

-- Check for duplicate emails (Expected: 2 - from our intentional duplicates)
SELECT SNOWFLAKE.CORE.DUPLICATE_COUNT(
  SELECT EMAIL FROM HORIZON_DEMO_DB.HEALTHCARE.PATIENTS
) AS duplicate_email_count;

In [ ]:
-- Custom DMF definition: validates SSN matches XXX-XX-XXXX
SELECT GET_DDL('FUNCTION', 'HORIZON_DEMO_DB.HEALTHCARE.INVALID_SSN_COUNT(TABLE(VARCHAR))');

## 5. Data Sharing & Agentic AI Distribution

### The Sharing Stack:
1. **Secure Views** enforce data policies even for consumers
2. **Shares** grant access to specific views without copying data
3. **Internal Marketplace** publishes data products with descriptions
4. **Cortex Agent** provides AI-powered natural language access to the data

### What we share:
- `REVENUE_DASHBOARD` - Aggregated revenue metrics (no PII)
- `DEPARTMENT_METRICS` - Department-level operational stats
- `HEALTHCARE_AGENT` - AI agent that answers analytics questions via natural language

In [ ]:
-- View the secure share contents
DESCRIBE SHARE HORIZON_DEMO_DB_HEALTHCARE_SHARE;

In [ ]:
-- Preview what consumers would see from the shared views
-- Revenue Dashboard: aggregated, no PII, minimum 5 patients per group
SELECT * FROM REVENUE_DASHBOARD
ORDER BY patient_count DESC
LIMIT 10;

In [ ]:
-- Department Metrics: operational view (also shared)
SELECT * FROM DEPARTMENT_METRICS
ORDER BY total_patients DESC;

In [ ]:
-- View the Cortex Agent
DESCRIBE AGENT HORIZON_DEMO_DB.HEALTHCARE.HEALTHCARE_AGENT;

In [ ]:
-- List semantic model on stage (backs the agent)
LIST @HORIZON_DEMO_DB.HEALTHCARE.SEMANTIC_MODELS;

## 6. Metadata Layer (Pipeline Context)

The metadata layer captures **pipeline execution details**, **medallion layer tags**, and **lineage** into a unified catalog that AI services can query.

### New Tags:
| Tag | Purpose |
|-----|---------|
| `PIPELINE_LAYER` | BRONZE / SILVER / GOLD |
| `SOURCE_SYSTEM` | EHR, BILLING_SYSTEM, CLAIMS_PORTAL |
| `LOAD_FREQUENCY` | REAL_TIME, HOURLY, DAILY, WEEKLY |
| `SLA_FRESHNESS_SECONDS` | Max acceptable staleness |

### Pipeline Execution Log:
Captures every run with source-to-target mapping, row counts, warehouse name/size, duration, and status.

In [ ]:
-- View tables by medallion layer
SELECT
  TABLE_NAME,
  pipeline_layer,
  LAST_PIPELINE_STATUS,
  PIPELINE_SUCCESS_RATE_PCT,
  AVG_LOAD_DURATION_SEC,
  WAREHOUSE_NAME,
  WAREHOUSE_SIZE,
  DOWNSTREAM_COUNT,
  STALENESS_SECONDS
FROM METADATA_CATALOG
ORDER BY
  CASE pipeline_layer WHEN 'BRONZE' THEN 1 WHEN 'SILVER' THEN 2 WHEN 'GOLD' THEN 3 ELSE 4 END,
  TABLE_NAME;

In [ ]:
-- Pipeline execution history with source-to-target mappings
SELECT
  pipeline_name,
  layer_from || ' -> ' || layer_to AS transition,
  source_table,
  target_table,
  rows_read,
  rows_written,
  rows_rejected,
  warehouse_name,
  warehouse_size,
  duration_seconds || 's' AS duration,
  status,
  execution_start
FROM PIPELINE_EXECUTION_LOG
ORDER BY execution_start DESC;

In [ ]:
-- Pipeline health by layer: success rate and avg duration
SELECT
  layer_to AS target_layer,
  COUNT(*) AS total_runs,
  SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS successes,
  SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failures,
  ROUND(SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS success_rate_pct,
  ROUND(AVG(duration_seconds), 1) AS avg_duration_sec,
  MAX(warehouse_name) AS warehouse,
  MAX(warehouse_size) AS wh_size
FROM PIPELINE_EXECUTION_LOG
GROUP BY layer_to
ORDER BY target_layer;

## 7. AI Context Integration

The metadata layer feeds into Snowflake AI services:
- **Cortex Agent** - multi-tool agent with healthcare analytics + metadata + search
- **Cortex Analyst** - semantic model over pipeline execution log and metadata catalog
- **Cortex Search** - natural language discovery of tables by description/tags

### Enhanced Agent: HEALTHCARE_METADATA_AGENT
Can answer both business questions AND metadata/ops questions:
- "What is the no-show rate by department?" (healthcare tool)
- "Which pipelines failed recently?" (metadata tool)
- "Find tables with billing data" (search tool)

In [ ]:
-- View the enhanced multi-tool agent
DESCRIBE AGENT HORIZON_DEMO_DB.HEALTHCARE.HEALTHCARE_METADATA_AGENT;

In [ ]:
-- View the Cortex Search service (enables NL table discovery)
SHOW CORTEX SEARCH SERVICES IN SCHEMA HORIZON_DEMO_DB.HEALTHCARE;

In [ ]:
-- Lineage cache: full data flow across all layers
SELECT
  anchor_table AS bronze_source,
  SOURCE_OBJECT_NAME || ' (' || SOURCE_OBJECT_DOMAIN || ')' AS from_object,
  TARGET_OBJECT_NAME || ' (' || TARGET_OBJECT_DOMAIN || ')' AS to_object,
  DISTANCE AS hops
FROM LINEAGE_CACHE
ORDER BY anchor_table, DISTANCE;

## Summary: Horizon Catalog Capabilities Demonstrated

| Capability | What We Built | Governance Benefit |
|-----------|---------------|-------------------|
| **Object Tagging** | PII_LEVEL, DATA_CLASSIFICATION, DATA_DOMAIN, RETENTION_DAYS | Automated discovery, compliance reporting, tag-based policy enforcement |
| **Masking Policy** | Tag-based dynamic masking (SSN, email, phone, address) | One policy scales to all PII columns; role-based visibility |
| **Row Access Policy** | Department-scoped patient records | Clinical staff sees only their patients |
| **Aggregation Policy** | Min group size of 5 on BILLING | Prevents re-identification from financial analytics |
| **Join Policy** | Analysts must join PATIENTS with another table | Prevents "SELECT * FROM PATIENTS" attacks |
| **Projection Policy** | Blocks medication name projection for analysts | Prevents linking prescriptions to patients |
| **Lineage** | 3-level pipeline with column-level tracking | Impact analysis, PII leakage detection, provenance |
| **Data Quality** | System + custom DMFs with bad data detection | Continuous monitoring visible in Snowsight UI |
| **Data Sharing** | Secure share with aggregated views | Zero-copy sharing with policies enforced |
| **Agentic AI** | Cortex Agent + semantic model | Natural language access to governed data |
| **Metadata Layer** | PIPELINE_EXECUTION_LOG + METADATA_CATALOG + layer tags | Pipeline health, freshness, source-to-target mapping |
| **AI Context** | Multi-tool agent + Cortex Search + metadata semantic model | AI answers business AND ops questions with full lineage context |

### Next Steps:
1. Navigate to **Snowsight > Data > Governance** to see policies and DMF results in the UI
2. Check **Data > Lineage** to visualize the pipeline graph
3. Test the agent in **Snowflake Intelligence** or via SQL
4. Add consumer accounts to the share for cross-account data sharing
5. Query the METADATA_CATALOG view to see unified pipeline health across all layers

In [ ]:
-- CLEANUP (uncomment to remove all demo objects)
-- DROP AGENT IF EXISTS HORIZON_DEMO_DB.HEALTHCARE.HEALTHCARE_AGENT;
-- DROP SHARE IF EXISTS HORIZON_DEMO_DB_HEALTHCARE_SHARE;
-- DROP DATABASE IF EXISTS HORIZON_DEMO_DB;